# 日线多尺度多头注意力模型：未来涨跌幅与涨跌方向预测

本 Notebook 在现有 A 股日线数据基础上，尝试使用多头注意力机制进行个股未来收益预测。由于目前只有日线数据，暂时使用不同长度的日线窗口来模拟不同频率的 K 线信息：过去 5 日代表短周期信息，过去 20 日代表中周期信息，过去 60 日代表长周期信息。

模型结构上，首先分别使用 Transformer Encoder 对短、中、长期日线序列进行编码，提取不同时间尺度下的价格、成交额、换手率、波动率和市场特征；然后通过 Multi-Head Attention 融合不同尺度的信息，最后输出两个预测结果：未来 1 日涨跌幅，以及未来 1 日是否上涨。

本实验的核心目标是回应“预测涨跌幅、预测涨或不涨”的量化研究问题。模型按照时间顺序划分 Train/Test：使用 2020--2024 年数据训练，在 2025--2026 年 Test 集上评估预测效果。主要评价指标包括 IC、Rank IC、方向准确率、AUC、Top 组收益和 Top-Bottom 多空收益。

该版本可以看作是多频 K 线 Transformer 模型的初步实现。后续如果获得分钟线、小时线或周线数据，可以将目前的 5/20/60 日多尺度输入替换为真实的多频 K 线输入，再进一步比较 Test 集预测效果。

In [ ]:
# Cell 1：导入包和路径

from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, roc_auc_score, mean_squared_error
from scipy.stats import spearmanr

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

PROJECT_DIR = Path("<LOCAL_V3_DIR>")

DATA_DIR = Path("<LOCAL_V1_DIR>/data_extracted/daily_temp3")

OUT_DIR = PROJECT_DIR / "attention_daily_output"
FIG_DIR = PROJECT_DIR / "figures"
REPORT_DIR = PROJECT_DIR / "reports"

for d in [OUT_DIR, FIG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
# Cell 2：读取现有日线数据

def read_csv_safely(path):
    for enc in ["utf-8", "utf-8-sig", "gbk", "gb18030"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)


def load_daily_data(data_dir):
    csv_files = sorted(data_dir.rglob("*.csv"))
    print("CSV数量:", len(csv_files))
    print("第一个文件:", csv_files[0])

    rows = []
    t0 = time.time()

    keep_cols = [
        "secID", "tradeDate",
        "preClosePrice", "openPrice", "highestPrice", "lowestPrice",
        "closePrice", "turnoverValue", "turnoverRate",
        "negMarketValue", "chgPct", "isOpen"
    ]

    for i, f in enumerate(csv_files):
        df = read_csv_safely(f)

        missing = [c for c in keep_cols if c not in df.columns]
        if missing:
            continue

        df = df[keep_cols].copy()
        df = df[df["isOpen"] == 1].copy()

        date = pd.to_datetime(f.stem, format="%Y%m%d", errors="coerce")
        if pd.isna(date):
            continue

        out = pd.DataFrame(index=df.index)
        out["date"] = date
        out["code"] = df["secID"].astype(str)

        out["ret"] = pd.to_numeric(df["chgPct"], errors="coerce")
        out["preclose"] = pd.to_numeric(df["preClosePrice"], errors="coerce")
        out["open"] = pd.to_numeric(df["openPrice"], errors="coerce")
        out["high"] = pd.to_numeric(df["highestPrice"], errors="coerce")
        out["low"] = pd.to_numeric(df["lowestPrice"], errors="coerce")
        out["close"] = pd.to_numeric(df["closePrice"], errors="coerce")
        out["amount"] = pd.to_numeric(df["turnoverValue"], errors="coerce")
        out["turnover"] = pd.to_numeric(df["turnoverRate"], errors="coerce")
        out["mcap"] = pd.to_numeric(df["negMarketValue"], errors="coerce")

        out = out.dropna(subset=["date", "code", "ret", "open", "high", "low", "close"])
        rows.append(out)

        if (i + 1) % 300 == 0:
            print(f"已读取 {i+1}/{len(csv_files)}，用时 {time.time() - t0:.1f} 秒")

    data = pd.concat(rows, ignore_index=True)
    data = data.sort_values(["code", "date"]).reset_index(drop=True)

    print("数据形状:", data.shape)
    print("日期范围:", data["date"].min(), "到", data["date"].max())
    print("股票数:", data["code"].nunique())

    return data


stock_df = load_daily_data(DATA_DIR)

In [ ]:
# Cell 3：构造日线 K 线特征

df = stock_df.copy()

# 收益率极端值处理
df["ret"] = pd.to_numeric(df["ret"], errors="coerce")
df = df[(df["ret"] > -0.25) & (df["ret"] < 0.25)].copy()

# K线特征
df["open_ret"] = (df["open"] - df["preclose"]) / df["preclose"].replace(0, np.nan)
df["close_ret"] = (df["close"] - df["open"]) / df["open"].replace(0, np.nan)
df["hl_range"] = (df["high"] - df["low"]) / df["preclose"].replace(0, np.nan)

df["amount_log"] = np.log1p(df["amount"])
df["turnover"] = df["turnover"]
df["mcap_log"] = np.log1p(df["mcap"])

# 滚动特征
df["ret_3"] = df.groupby("code")["ret"].transform(lambda x: x.rolling(3).sum())
df["ret_5"] = df.groupby("code")["ret"].transform(lambda x: x.rolling(5).sum())
df["ret_20"] = df.groupby("code")["ret"].transform(lambda x: x.rolling(20).sum())
df["vol_5"] = df.groupby("code")["ret"].transform(lambda x: x.rolling(5).std())
df["vol_20"] = df.groupby("code")["ret"].transform(lambda x: x.rolling(20).std())

# 市场特征
market = df.groupby("date").agg(
    market_ret=("ret", "mean"),
    market_up_ratio=("ret", lambda x: (x > 0).mean()),
    market_vol=("ret", "std"),
    market_amount=("amount", "sum"),
).reset_index()

market["market_ret_5"] = market["market_ret"].rolling(5).sum()
market["market_ret_20"] = market["market_ret"].rolling(20).sum()
market["market_vol_20"] = market["market_ret"].rolling(20).std()
market["market_amount_log"] = np.log1p(market["market_amount"])

df = df.merge(market, on="date", how="left")

feature_cols = [
    "ret",
    "open_ret",
    "close_ret",
    "hl_range",
    "amount_log",
    "turnover",
    "mcap_log",
    "ret_3",
    "ret_5",
    "ret_20",
    "vol_5",
    "vol_20",
    "market_ret",
    "market_up_ratio",
    "market_vol",
    "market_ret_5",
    "market_ret_20",
    "market_vol_20",
    "market_amount_log",
]

df = df.replace([np.inf, -np.inf], np.nan)
df = df.sort_values(["code", "date"]).reset_index(drop=True)

print("特征数量:", len(feature_cols))
print("数据形状:", df.shape)
df[["date", "code"] + feature_cols].head()

In [ ]:
# Cell 4：构造预测目标

# 预测未来1日收益率
df["target_ret_1d"] = df.groupby("code")["ret"].shift(-1)

# 预测未来1日是否上涨
df["target_up_1d"] = np.where(
    df["target_ret_1d"].notna(),
    (df["target_ret_1d"] > 0).astype(int),
    np.nan
)

df = df.dropna(subset=["target_ret_1d", "target_up_1d"]).copy()

df = df[
    (df["target_ret_1d"] > -0.25) &
    (df["target_ret_1d"] < 0.25)
].copy()

print("构造 target 后:", df.shape)
print("日期范围:", df["date"].min(), "到", df["date"].max())
df[["date", "code", "ret", "target_ret_1d", "target_up_1d"]].head()

In [ ]:
# Cell 5：Train/Test 切分和标准化

train_mask = df["date"] <= pd.Timestamp("2024-12-31")
test_mask = df["date"] >= pd.Timestamp("2025-01-01")

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

print("Train:", train_df["date"].min(), "到", train_df["date"].max(), train_df.shape)
print("Test :", test_df["date"].min(), "到", test_df["date"].max(), test_df.shape)

# 用训练集统计量做标准化，避免信息泄露
medians = train_df[feature_cols].median()
stds = train_df[feature_cols].std().replace(0, np.nan)
stds = stds.fillna(1)

df_scaled = df.copy()

for c in feature_cols:
    df_scaled[c] = pd.to_numeric(df_scaled[c], errors="coerce")
    df_scaled[c] = df_scaled[c].fillna(medians[c])
    df_scaled[c] = (df_scaled[c] - medians[c]) / stds[c]

df_scaled = df_scaled.replace([np.inf, -np.inf], 0).fillna(0)

target_scale = train_df["target_ret_1d"].std()
print("target_scale:", target_scale)

In [ ]:
# Cell 6：构造多尺度序列 Dataset

class MultiScaleDailyDataset(Dataset):
    def __init__(
        self,
        data,
        feature_cols,
        start_date,
        end_date,
        short_window=5,
        mid_window=20,
        long_window=60,
        max_samples=None,
        random_state=42,
    ):
        self.data = data.sort_values(["code", "date"]).reset_index(drop=True)
        self.feature_cols = feature_cols

        self.short_window = short_window
        self.mid_window = mid_window
        self.long_window = long_window

        self.groups = {}
        self.samples = []

        for code, g in self.data.groupby("code"):
            g = g.sort_values("date").reset_index(drop=True)

            x = g[feature_cols].values.astype(np.float32)
            y_ret = g["target_ret_1d"].values.astype(np.float32)
            y_up = g["target_up_1d"].values.astype(np.float32)
            dates = pd.to_datetime(g["date"]).values

            self.groups[code] = {
                "x": x,
                "y_ret": y_ret,
                "y_up": y_up,
                "dates": dates,
            }

            for i in range(long_window - 1, len(g)):
                date_i = pd.Timestamp(dates[i])

                if date_i < pd.Timestamp(start_date) or date_i > pd.Timestamp(end_date):
                    continue

                if np.isnan(y_ret[i]) or np.isnan(y_up[i]):
                    continue

                self.samples.append((code, i))

        if max_samples is not None and len(self.samples) > max_samples:
            rng = np.random.default_rng(random_state)
            idx = rng.choice(len(self.samples), size=max_samples, replace=False)
            self.samples = [self.samples[i] for i in idx]

        print(f"{start_date} 到 {end_date} 样本数:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        code, i = self.samples[idx]
        g = self.groups[code]
        x = g["x"]

        short_seq = x[i - self.short_window + 1 : i + 1]
        mid_seq = x[i - self.mid_window + 1 : i + 1]
        long_seq = x[i - self.long_window + 1 : i + 1]

        y_ret = g["y_ret"][i] / target_scale
        y_up = g["y_up"][i]

        date = str(pd.Timestamp(g["dates"][i]).date())

        return {
            "short": torch.tensor(short_seq, dtype=torch.float32),
            "mid": torch.tensor(mid_seq, dtype=torch.float32),
            "long": torch.tensor(long_seq, dtype=torch.float32),
            "y_ret": torch.tensor(y_ret, dtype=torch.float32),
            "y_up": torch.tensor(y_up, dtype=torch.float32),
            "date": date,
            "code": code,
        }


train_dataset = MultiScaleDailyDataset(
    df_scaled,
    feature_cols,
    start_date="2020-01-01",
    end_date="2024-12-31",
    short_window=5,
    mid_window=20,
    long_window=60,
    max_samples=300000,
)

test_dataset = MultiScaleDailyDataset(
    df_scaled,
    feature_cols,
    start_date="2025-01-01",
    end_date="2026-12-31",
    short_window=5,
    mid_window=20,
    long_window=60,
    max_samples=80000,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
)

In [ ]:
# Cell 7：多头注意力模型

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).float().unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        if d_model % 2 == 0:
            pe[:, 1::2] = torch.cos(position * div_term)
        else:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ScaleEncoder(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, num_layers=1, dropout=0.1):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        x = self.encoder(x)

        # 用最后一个时间点表示当前状态
        return x[:, -1, :]


class MultiScaleAttentionModel(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=4, dropout=0.1):
        super().__init__()

        self.short_encoder = ScaleEncoder(input_dim, d_model, nhead, num_layers=1, dropout=dropout)
        self.mid_encoder = ScaleEncoder(input_dim, d_model, nhead, num_layers=1, dropout=dropout)
        self.long_encoder = ScaleEncoder(input_dim, d_model, nhead, num_layers=1, dropout=dropout)

        self.scale_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 2),
        )

    def forward(self, short, mid, long):
        short_emb = self.short_encoder(short)
        mid_emb = self.mid_encoder(mid)
        long_emb = self.long_encoder(long)

        # batch, 3, d_model
        scale_tokens = torch.stack([short_emb, mid_emb, long_emb], dim=1)

        attn_out, attn_weights = self.scale_attention(
            scale_tokens,
            scale_tokens,
            scale_tokens,
            need_weights=True,
        )

        fused = self.norm(attn_out.mean(dim=1))

        out = self.head(fused)

        pred_ret_scaled = out[:, 0]
        pred_up_logit = out[:, 1]

        return pred_ret_scaled, pred_up_logit, attn_weights


model = MultiScaleAttentionModel(
    input_dim=len(feature_cols),
    d_model=64,
    nhead=4,
    dropout=0.1,
).to(DEVICE)

model

In [ ]:
# Cell 8：训练函数和预测函数

def train_one_epoch(model, loader, optimizer):
    model.train()

    mse = nn.MSELoss()
    bce = nn.BCEWithLogitsLoss()

    total_loss = 0
    n = 0

    for batch in loader:
        short = batch["short"].to(DEVICE)
        mid = batch["mid"].to(DEVICE)
        long = batch["long"].to(DEVICE)

        y_ret = batch["y_ret"].to(DEVICE)
        y_up = batch["y_up"].to(DEVICE)

        pred_ret_scaled, pred_up_logit, _ = model(short, mid, long)

        loss_reg = mse(pred_ret_scaled, y_ret)
        loss_cls = bce(pred_up_logit, y_up)

        loss = loss_reg + 0.5 * loss_cls

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        bs = len(y_ret)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / n


@torch.no_grad()
def predict(model, loader):
    model.eval()

    rows = []

    for batch in loader:
        short = batch["short"].to(DEVICE)
        mid = batch["mid"].to(DEVICE)
        long = batch["long"].to(DEVICE)

        pred_ret_scaled, pred_up_logit, attn_weights = model(short, mid, long)

        pred_ret = pred_ret_scaled.cpu().numpy() * target_scale
        pred_up_prob = torch.sigmoid(pred_up_logit).cpu().numpy()

        y_ret = batch["y_ret"].cpu().numpy() * target_scale
        y_up = batch["y_up"].cpu().numpy()

        for i in range(len(y_ret)):
            rows.append({
                "date": batch["date"][i],
                "code": batch["code"][i],
                "target_ret_1d": y_ret[i],
                "target_up_1d": y_up[i],
                "pred_ret": pred_ret[i],
                "pred_up_prob": pred_up_prob[i],
            })

    return pd.DataFrame(rows)

In [ ]:
# Cell 9：评估函数

def calc_metrics(pred_df):
    d = pred_df.dropna().copy()
    d["pred_up"] = (d["pred_up_prob"] >= 0.5).astype(int)

    ic = d["pred_ret"].corr(d["target_ret_1d"])
    rank_ic = spearmanr(d["pred_ret"], d["target_ret_1d"]).correlation

    acc = accuracy_score(d["target_up_1d"], d["pred_up"])

    try:
        auc = roc_auc_score(d["target_up_1d"], d["pred_up_prob"])
    except Exception:
        auc = np.nan

    mse = mean_squared_error(d["target_ret_1d"], d["pred_ret"])

    rows = []

    for date, g in d.groupby("date"):
        if len(g) < 50:
            continue

        rows.append({
            "date": date,
            "daily_ic": g["pred_ret"].corr(g["target_ret_1d"]),
            "daily_rank_ic": spearmanr(g["pred_ret"], g["target_ret_1d"]).correlation,
            "n": len(g),
        })

    daily_ic_df = pd.DataFrame(rows)

    return {
        "IC": ic,
        "Rank_IC": rank_ic,
        "Daily_IC_Mean": daily_ic_df["daily_ic"].mean(),
        "Daily_Rank_IC_Mean": daily_ic_df["daily_rank_ic"].mean(),
        "Direction_Accuracy": acc,
        "AUC": auc,
        "MSE": mse,
        "N": len(d),
        "N_Days": d["date"].nunique(),
    }, daily_ic_df


def group_return(pred_df, n_groups=5):
    rows = []

    for date, g in pred_df.groupby("date"):
        if len(g) < n_groups * 20:
            continue

        temp = g.copy()
        temp["group"] = pd.qcut(
            temp["pred_ret"].rank(method="first"),
            n_groups,
            labels=False,
        ) + 1

        grp = temp.groupby("group")["target_ret_1d"].mean()

        row = {"date": date}
        for i in range(1, n_groups + 1):
            row[f"g{i}"] = grp.get(i, np.nan)

        row["top"] = row[f"g{n_groups}"]
        row["long_short"] = row[f"g{n_groups}"] - row["g1"]
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Cell 10：训练模型

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

EPOCHS = 6

best_rank_ic = -999
best_pred = None
best_metrics = None
best_daily_ic = None

for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, optimizer)

    pred_test = predict(model, test_loader)
    metrics, daily_ic_df = calc_metrics(pred_test)

    print(
        f"Epoch {epoch} | "
        f"Loss {loss:.6f} | "
        f"IC {metrics['IC']:.4f} | "
        f"RankIC {metrics['Rank_IC']:.4f} | "
        f"DailyRankIC {metrics['Daily_Rank_IC_Mean']:.4f} | "
        f"Acc {metrics['Direction_Accuracy']:.4f} | "
        f"AUC {metrics['AUC']:.4f}"
    )

    if metrics["Daily_Rank_IC_Mean"] > best_rank_ic:
        best_rank_ic = metrics["Daily_Rank_IC_Mean"]
        best_pred = pred_test.copy()
        best_metrics = metrics.copy()
        best_daily_ic = daily_ic_df.copy()

In [ ]:
# Cell 11：保存结果

pred_test = best_pred.copy()
metrics = best_metrics.copy()
daily_ic_df = best_daily_ic.copy()

grp_df = group_return(pred_test, n_groups=5)

metrics["Top_Group_Mean_Return"] = grp_df["top"].mean()
metrics["Long_Short_Mean_Return"] = grp_df["long_short"].mean()
metrics["Long_Short_Daily_Sharpe"] = grp_df["long_short"].mean() / grp_df["long_short"].std()

pred_test.to_csv(
    OUT_DIR / "attention_daily_test_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

pd.DataFrame([metrics]).to_csv(
    OUT_DIR / "attention_daily_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

daily_ic_df.to_csv(
    OUT_DIR / "attention_daily_ic.csv",
    index=False,
    encoding="utf-8-sig",
)

grp_df.to_csv(
    OUT_DIR / "attention_daily_group_return.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Test metrics:")
display(pd.DataFrame([metrics]))

print("结果保存到:", OUT_DIR)

In [ ]:
# Cell 12：画分组收益图

grp_plot = grp_df.copy()
grp_plot["date"] = pd.to_datetime(grp_plot["date"])

grp_plot["cum_top"] = (1 + grp_plot["top"].fillna(0)).cumprod() - 1
grp_plot["cum_long_short"] = (1 + grp_plot["long_short"].fillna(0)).cumprod() - 1

plt.figure(figsize=(12, 5))

plt.plot(grp_plot["date"], grp_plot["cum_top"], label="Top Group")
plt.plot(grp_plot["date"], grp_plot["cum_long_short"], label="Top-Bottom Long Short")

plt.title("Daily Multi-Scale Multi-Head Attention Model")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

fig_path = FIG_DIR / "attention_daily_cumret.png"
plt.savefig(fig_path, dpi=200)
plt.show()

print("图片保存到:", fig_path)

In [ ]:
# Cell 13：画 Daily Rank IC

ic_plot = daily_ic_df.copy()
ic_plot["date"] = pd.to_datetime(ic_plot["date"])

plt.figure(figsize=(12, 4))

plt.plot(ic_plot["date"], ic_plot["daily_rank_ic"])
plt.axhline(0, linestyle="--", linewidth=1)

plt.title("Daily Rank IC: Multi-Scale Attention Model")
plt.xlabel("Date")
plt.ylabel("Daily Rank IC")
plt.grid(alpha=0.3)
plt.tight_layout()

fig_path = FIG_DIR / "attention_daily_rank_ic.png"
plt.savefig(fig_path, dpi=200)
plt.show()

print("图片保存到:", fig_path)

In [ ]:
# Cell 14：生成简要报告

report = f"""
# 日线多尺度多头注意力模型 Test 集预测报告

## 一、任务说明

本版本在现有日线数据基础上，尝试使用多头注意力机制进行个股未来收益预测。由于当前只有日线数据，暂时使用不同长度的日线窗口模拟不同频率信息：

- 过去 5 日：短周期 K 线信息
- 过去 20 日：中周期 K 线信息
- 过去 60 日：长周期 K 线信息

模型通过 Transformer Encoder 分别编码不同周期的日线序列，再用 Multi-Head Attention 融合短、中、长期信息，最后预测未来 1 日涨跌幅和未来 1 日是否上涨。

## 二、Train/Test 切分

训练集：2020-01-01 至 2024-12-31  
测试集：2025-01-01 至 2026-02-02  

## 三、Test 集预测效果

- IC：{metrics['IC']:.4f}
- Rank IC：{metrics['Rank_IC']:.4f}
- Daily IC Mean：{metrics['Daily_IC_Mean']:.4f}
- Daily Rank IC Mean：{metrics['Daily_Rank_IC_Mean']:.4f}
- 方向准确率：{metrics['Direction_Accuracy']:.4f}
- AUC：{metrics['AUC']:.4f}
- Top 组日均收益：{metrics['Top_Group_Mean_Return']:.6f}
- Top-Bottom 日均收益：{metrics['Long_Short_Mean_Return']:.6f}
- Top-Bottom 日频 Sharpe：{metrics['Long_Short_Daily_Sharpe']:.4f}

## 四、结果解释

该版本是对老师提出的“多头注意力机制、多频 K 线预测未来日线”的初步实现。由于当前只有日线数据，所以先用 5 日、20 日、60 日三个不同时间尺度近似不同频率 K 线。后续如果有分钟线、小时线或周线数据，可以将不同频率的真实 K 线序列输入模型，再比较 Test 集上的 IC、Rank IC、方向准确率、AUC 和分组收益表现。
"""

report_path = REPORT_DIR / "attention_daily_report.md"
report_path.write_text(report, encoding="utf-8")

print(report)
print("报告保存到:", report_path)